In [7]:
import json
import requests

from requests.auth import HTTPBasicAuth

In [3]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

JIRA_URL = os.getenv("JIRA_URL")
JIRA_EMAIL = os.getenv("JIRA_EMAIL")
JIRA_API_TOKEN = os.getenv("JIRA_API_TOKEN")
PROJECT_KEY=os.getenv("JIRA_PROJECT_KEY")

response = requests.get(
    f"{JIRA_URL}/rest/api/3/myself",
    auth=(JIRA_EMAIL, JIRA_API_TOKEN),
    headers={"Accept": "application/json"},
)

print("Status code:", response.status_code)
print(response.json())

Status code: 200
{'self': 'https://sakshiborage19.atlassian.net/rest/api/3/user?accountId=712020:299d9613-81f8-43c9-ae14-af433ffe6422', 'accountId': '712020:299d9613-81f8-43c9-ae14-af433ffe6422', 'accountType': 'atlassian', 'emailAddress': 'sakshiborage19@gmail.com', 'avatarUrls': {'48x48': 'https://secure.gravatar.com/avatar/3b0a48c187c09bc9d20f5f3f37bf2cef?d=https%3A%2F%2Favatar-management--avatars.us-west-2.prod.public.atl-paas.net%2Finitials%2FSB-4.png', '24x24': 'https://secure.gravatar.com/avatar/3b0a48c187c09bc9d20f5f3f37bf2cef?d=https%3A%2F%2Favatar-management--avatars.us-west-2.prod.public.atl-paas.net%2Finitials%2FSB-4.png', '16x16': 'https://secure.gravatar.com/avatar/3b0a48c187c09bc9d20f5f3f37bf2cef?d=https%3A%2F%2Favatar-management--avatars.us-west-2.prod.public.atl-paas.net%2Finitials%2FSB-4.png', '32x32': 'https://secure.gravatar.com/avatar/3b0a48c187c09bc9d20f5f3f37bf2cef?d=https%3A%2F%2Favatar-management--avatars.us-west-2.prod.public.atl-paas.net%2Finitials%2FSB-4.png

In [2]:
ticket_data = {
    "summary": "Passenger baggage not received",

    "description": """

Issue :
Passenger reported that checked baggage was not available at the destination airport.

Priority :
High

Department :
Baggage Services
"""
}

In [4]:
ISSUE_TYPE="Task"

In [5]:
payload = {
    "fields": {

        "project": {
            "key": PROJECT_KEY
        },

        "summary": ticket_data["summary"],

        "description": {
            "type": "doc",
            "version": 1,
            "content": [
                {
                    "type": "paragraph",
                    "content": [
                        {
                            "type": "text",
                            "text": ticket_data["description"]
                        }
                    ]
                }
            ]
        },

        "issuetype": {
            "name": ISSUE_TYPE
        }
    }
}

In [9]:
url = f"{JIRA_URL}/rest/api/3/issue"

response = requests.post(
    url,
    auth=HTTPBasicAuth(JIRA_EMAIL, JIRA_API_TOKEN),
    headers={
        "Accept": "application/json",
        "Content-Type": "application/json"
    },
    data=json.dumps(payload)
)

In [10]:
print("Status Code:", response.status_code)

print(response.text)

Status Code: 201
{"id":"10046","key":"AIR-2","self":"https://sakshiborage19.atlassian.net/rest/api/3/issue/10046"}


In [11]:

import os
import requests
from dotenv import load_dotenv

load_dotenv()

JIRA_URL = os.getenv("JIRA_URL")
JIRA_EMAIL = os.getenv("JIRA_EMAIL")
JIRA_API_TOKEN = os.getenv("JIRA_API_TOKEN")
JIRA_PROJECT_KEY = os.getenv("JIRA_PROJECT_KEY")

# Adjust this to match an issue type that actually exists in your project -
# Company-managed projects created from different templates have different
# default issue types (e.g. "Task", "Bug", "Story", or for service-desk
# style templates: "Incident", "Service Request"). Check your project's
# issue type scheme if this doesn't match and you get a 400 error.
ISSUE_TYPE_NAME = "Task"


def build_description_adf(reasoning: str, original_message: str) -> dict:
    """
    Jira Cloud REST API v3 requires the description field in Atlassian
    Document Format (ADF), not a plain string. This builds a structured
    description with the AI's reasoning and the original message in
    clearly separated, headed sections - easier for a human agent to scan
    than one concatenated block of text.
    """
    return {
        "type": "doc",
        "version": 1,
        "content": [
            {
                "type": "heading",
                "attrs": {"level": 3},
                "content": [{"type": "text", "text": "AI Classification Reasoning"}],
            },
            {
                "type": "paragraph",
                "content": [{"type": "text", "text": reasoning}],
            },
            {
                "type": "heading",
                "attrs": {"level": 3},
                "content": [{"type": "text", "text": "Original Message"}],
            },
            {
                "type": "paragraph",
                "content": [{"type": "text", "text": original_message}],
            },
        ],
    }


def create_jira_ticket(
    category: str,
    priority: str,
    assigned_team: str,
    reasoning: str,
    original_message: str,
) -> dict:
    """
    Creates a Jira issue from a classification result.

    Returns {"key": ..., "url": ...} on success.
    Raises requests.HTTPError on failure - the caller is expected to wrap
    this with the same retry/fallback pattern used for other API calls in
    the pipeline (see the API-failure-handling discussion), not call it
    unguarded.
    """
    readable_category = category.replace("_", " ").title()
    summary = f"[{priority}] {readable_category}: {original_message[:50]}"

    payload = {
        "fields": {
            "project": {"key": JIRA_PROJECT_KEY},
            "summary": summary,
            "description": build_description_adf(reasoning, original_message),
            "issuetype": {"name": ISSUE_TYPE_NAME},
            "priority": {"name": priority},  # High / Medium / Low - matches Jira's default scheme
            "components": [{"name": assigned_team}],
            "labels": [category],  # e.g. "fraud_security" - extra filter, independent of Component
        }
    }

    response = requests.post(
        f"{JIRA_URL}/rest/api/3/issue",
        json=payload,
        auth=(JIRA_EMAIL, JIRA_API_TOKEN),
        headers={"Content-Type": "application/json"},
    )

    response.raise_for_status()
    result = response.json()
    return {
        "key": result["key"],
        "url": f"{JIRA_URL}/browse/{result['key']}",
    }


if __name__ == "__main__":
    # One hardcoded test case - confirm this creates a real ticket before
    # connecting it to the classification pipeline.
    test_result = create_jira_ticket(
        category="baggage",
        priority="High",
        assigned_team="Baggage Services",
        reasoning="Unresolved lost baggage after repeated contact attempts; escalated due to delay and urgent tone.",
        original_message=(
            "THIS IS ABSOLUTELY UNACCEPTABLE!! I've called FOUR times and "
            "nobody can tell me where my suitcase is!!!"
        ),
    )
    print("Created ticket:", test_result)

Created ticket: {'key': 'AIR-3', 'url': 'https://sakshiborage19.atlassian.net/browse/AIR-3'}
